In [1]:
import pandas as pd
import numpy as np
from functools import reduce
from urllib.parse import quote


class DataProcessor:
    """Classe pour gérer la récupération et le traitement de données depuis une API."""
    
    def __init__(self, url_base, tags_other, tags_selected, start, end, 
                 interval='PT20M', hS='00', hF='23', cred_file="../../../cred.txt"):
        """
        Initialise le processeur de données.
        
        Parameters:
        url_base : str, URL de base de l'API
        tags_other : list, liste des tags autres
        tags_selected : list, liste des tags sélectionnés
        start : str, date de début (format 'YYYY-MM-DD')
        end : str, date de fin (format 'YYYY-MM-DD')
        interval : str, intervalle d'agrégation (défaut: 'PT20M')
        hS : str, heure de début (défaut: '00')
        hF : str, heure de fin (défaut: '23')
        cred_file : str, chemin vers le fichier de credentials
        """
        self.url_base = url_base
        self.tags_other = tags_other
        self.tags_selected = tags_selected
        self.start = start
        self.end = end
        self.interval = interval
        self.hS = hS
        self.hF = hF
        self.credentials = self._read_cred(cred_file)
        
        # DataFrames
        self.df = None  # DataFrame brut fusionné
        self.data = None  # Copie de travail
        
    def _read_cred(self, cred_file):
        """
        Lit les credentials depuis un fichier.
        
        Parameters:
        cred_file : str, chemin vers le fichier
        
        Returns:
        str : contenu du fichier credentials
        """
        with open(cred_file, "r") as f:
            cred = f.read()
        return cred
    
    def get_OI(self, tag):
        """
        Récupère les données depuis l'API pour un tag donné.
        
        Parameters:
        tag : str, tag à récupérer
        
        Returns:
        list : valeurs récupérées
        """
        url_all = (f"{self.url_base}data-reference={tag}&aggregation=TIME"
                   f"&aggregation-function=MEAN&from={self.start}T{self.hS}%3A00%3A00.000Z"
                   f"&to={self.end}T{self.hF}%3A59%3A59.000Z&aggregation-period={self.interval}")
        
        d_data = pd.read_json(url_all, storage_options={'Authorization': 'basic ' + self.credentials})
        arr = np.asarray(np.asarray(d_data['values'])[0])
        return d_data['values'][0]
    
    def get_data(self):
        """
        Récupère les données pour tous les tags.
        
        Returns:
        list : liste de DataFrames
        """
        tags = self.tags_other + self.tags_selected
        liste = []
        for tag in tags:
            urlTag = quote(tag, safe=':/?#[]@!$&\'()*+,;=')
            data = self.get_OI(urlTag)
            df_temp = pd.DataFrame(data)
            df_temp['timestamp'] = pd.to_datetime(df_temp['timestamp'])
            df_temp = df_temp.set_index('timestamp')
            df_temp = df_temp.rename(columns={'value': tag})
            liste.append(df_temp)
        return liste
    
    def merge(self):
        """
        Fusionne les données récupérées dans self.df.
        Crée également une copie de travail dans self.data.
        """
        df_list = self.get_data()
        self.df = reduce(lambda left, right: pd.merge(left, right, left_index=True, 
                                                       right_index=True, how='outer'), df_list)
        self.data = self.df.copy()
        print(f"Données chargées : {len(self.df)} lignes, {len(self.df.columns)} colonnes")
    
    def read(self, start=None, end=None, interval=None, hS=None, hF=None):
        """
        Réinitialise et recharge le DataFrame avec de nouveaux paramètres.
        
        Parameters:
        start : str, date de début (optionnel)
        end : str, date de fin (optionnel)
        interval : str, intervalle d'agrégation (optionnel)
        hS : str, heure de début (optionnel)
        hF : str, heure de fin (optionnel)
        """
        # Mise à jour des paramètres si fournis
        if start is not None:
            self.start = start
        if end is not None:
            self.end = end
        if interval is not None:
            self.interval = interval
        if hS is not None:
            self.hS = hS
        if hF is not None:
            self.hF = hF
        
        # Réinitialisation et rechargement
        self.df = None
        self.data = None
        self.merge()
    
    def filtering(self, tag, min_val, max_val, na=None):
        """
        Applique des filtres sur self.data.
        
        Parameters:
        tag : str or list, tag(s) à filtrer
        min_val : float or list, valeur(s) minimale(s)
        max_val : float or list, valeur(s) maximale(s)
        na : str or list or None, tag(s) sur lequel appliquer dropna
        
        Returns:
        DataProcessor : self pour chaînage
        """
        # Supprime les lignes entièrement vides
        self.data = self.data.dropna(how="all")
        
        # Convertit tag en liste si c'est une chaîne
        if isinstance(tag, str):
            tags = [tag]
            min_vals = [min_val]
            max_vals = [max_val]
        else:
            tags = tag
            min_vals = min_val if isinstance(min_val, list) else [min_val] * len(tags)
            max_vals = max_val if isinstance(max_val, list) else [max_val] * len(tags)
        
        # Applique les filtres min/max pour chaque tag
        for t, min_v, max_v in zip(tags, min_vals, max_vals):
            self.data = self.data[(self.data[t] > min_v) & (self.data[t] < max_v)]
        
        # Applique dropna si spécifié
        if na is not None:
            na_list = [na] if isinstance(na, str) else na
            self.data = self.data.dropna(how="all", subset=na_list)
        
        print(f"Après filtrage : {len(self.data)} lignes")
        return self
    
    def ajoute_cumul(self, col_poids, col_valeur, ratio, nom):
        """
        Ajoute une colonne de cumul à self.data.
        
        Parameters:
        col_poids : str, nom de la colonne de poids
        col_valeur : str, nom de la colonne de valeur
        ratio : float, ratio de division
        nom : str, nom de la nouvelle colonne
        
        Returns:
        DataProcessor : self pour chaînage
        """
        self.data[nom] = (self.data[col_poids] * self.data[col_valeur]) / ratio
        print(f"Colonne '{nom}' ajoutée")
        return self
    
    def ajouter_moyennes_glissantes(self, col_poids, col_valeur, nom, window=10):
        """
        Ajoute une colonne de moyenne pondérée glissante à self.data.
        
        Parameters:
        col_poids : str, nom de la colonne de poids
        col_valeur : str, nom de la colonne de valeurs
        nom : str, nom de la nouvelle colonne
        window : int, taille de la fenêtre glissante
        
        Returns:
        DataProcessor : self pour chaînage
        """
        poids = self.data[col_poids]
        valeurs = self.data[col_valeur]
        
        numerateur = (poids * valeurs).rolling(window=window).sum()
        denominateur = poids.rolling(window=window).sum()
        self.data[nom] = numerateur / denominateur
        
        print(f"Colonne '{nom}' ajoutée (moyenne glissante sur {window} valeurs)")
        return self
    
    def reset_data(self):
        """
        Réinitialise self.data à partir de self.df.
        """
        if self.df is not None:
            self.data = self.df.copy()
            print("self.data réinitialisé à partir de self.df")
        else:
            print("Aucune donnée brute disponible (self.df est None)")
    
    # Setters
    def set_url_base(self, url_base):
        """Modifie l'URL de base."""
        self.url_base = url_base
        print(f"URL de base modifiée : {url_base}")
    
    def set_tags_other(self, tags_other):
        """Modifie la liste des tags autres."""
        self.tags_other = tags_other
        print(f"Tags autres modifiés : {tags_other}")
    
    def set_tags_selected(self, tags_selected):
        """Modifie la liste des tags sélectionnés."""
        self.tags_selected = tags_selected
        print(f"Tags sélectionnés modifiés : {tags_selected}")
    
    def set_start(self, start):
        """Modifie la date de début."""
        self.start = start
        print(f"Date de début modifiée : {start}")
    
    def set_end(self, end):
        """Modifie la date de fin."""
        self.end = end
        print(f"Date de fin modifiée : {end}")
    
    def set_interval(self, interval):
        """Modifie l'intervalle d'agrégation."""
        self.interval = interval
        print(f"Intervalle modifié : {interval}")
    
    def set_hS(self, hS):
        """Modifie l'heure de début."""
        self.hS = hS
        print(f"Heure de début modifiée : {hS}")
    
    def set_hF(self, hF):
        """Modifie l'heure de fin."""
        self.hF = hF
        print(f"Heure de fin modifiée : {hF}")
    
    def info(self):
        """
        Affiche tous les paramètres de la classe.
        """
        print("="*60)
        print("INFORMATIONS DataProcessor")
        print("="*60)
        print(f"URL de base       : {self.url_base}")
        print(f"Tags autres       : {self.tags_other}")
        print(f"Tags sélectionnés : {self.tags_selected}")
        print(f"Tous les tags     : {self.tags_other + self.tags_selected}")
        print(f"Date de début     : {self.start}")
        print(f"Date de fin       : {self.end}")
        print(f"Intervalle        : {self.interval}")
        print(f"Heure de début    : {self.hS}")
        print(f"Heure de fin      : {self.hF}")
        print("-"*60)
        if self.df is not None:
            print(f"DataFrame brut (df)   : {len(self.df)} lignes, {len(self.df.columns)} colonnes")
            print(f"Colonnes df           : {list(self.df.columns)}")
        else:
            print("DataFrame brut (df)   : Non chargé")
        
        if self.data is not None:
            print(f"DataFrame travail (data) : {len(self.data)} lignes, {len(self.data.columns)} colonnes")
            print(f"Colonnes data            : {list(self.data.columns)}")
        else:
            print("DataFrame travail (data) : Non chargé")
        print("="*60)


In [ ]:
urlBase = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?'
tags_other = ['CTY_A1000M_Poids container','CTY_A1000M_Teneur arr. Vit. A (UV)','CTY_A1000M_Titre VA AC','CTY_A1000M_Numéro Container','CTY_A1000R_Poids container','CTY_A1000R_Teneur arr. Vit. A (UV)','CTY_A1000R_Titre VA AC','CTY_A1000R_Numéro Container']; #,'CTY_FHA1000M_Teneur_VA']

start = '2010-01-01'
end = '2025-12-16'
df_list = get_data(tags,start,end)
data = merge_data(df_list)
data_tags = ['A1000M_Poids','A1000M_UV_Auto','A1000M_Labo','A1000M_Num','A1000R_Poids','A1000R_UV_Auto','A1000R_Labo','A1000R_Num']
brute = data.copy()
brute.columns = data_tags
brute.describe()

In [4]:
# Initialisation
processor = DataProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    tags_other = [],
    tags_selected = ['CTY_A1000M_Poids container','CTY_A1000M_Teneur arr. Vit. A (UV)','CTY_A1000M_Titre VA AC','CTY_A1000M_Numéro Container','CTY_A1000R_Poids container','CTY_A1000R_Teneur arr. Vit. A (UV)','CTY_A1000R_Titre VA AC','CTY_A1000R_Numéro Container'],
    start='2010-01-01',
    end='2025-12-16'
)

# Afficher les infos
processor.info()

# Charger les données
processor.merge()

# Afficher les infos après chargement
processor.info()

# Appliquer des filtres (chaînage possible)
processor.filtering(
    tag=['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo'],
    min_val=[700, 800_000, 700_000],
    max_val=[1300, 1_300_000, 1_300_000],
    na='A1000M_Poids'
)

# Ajouter des colonnes
processor.ajoute_cumul('A1000M_Poids', 'A1000M_UV_Auto', 1000, 'Cumul_UV')
processor.ajouter_moyennes_glissantes('A1000M_Poids', 'A1000M_Labo', 'Moyenne_Labo', window=10)

# Modifier des paramètres
processor.set_start('2020-01-01')
processor.set_interval('PT30M')

# Recharger avec les nouveaux paramètres
processor.read()

# Réinitialiser data à partir de df
processor.reset_data()

# Afficher les infos finales
processor.info()

INFORMATIONS DataProcessor
URL de base       : https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?
Tags autres       : []
Tags sélectionnés : ['CTY_A1000M_Poids container', 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'CTY_A1000M_Titre VA AC', 'CTY_A1000M_Numéro Container', 'CTY_A1000R_Poids container', 'CTY_A1000R_Teneur arr. Vit. A (UV)', 'CTY_A1000R_Titre VA AC', 'CTY_A1000R_Numéro Container']
Tous les tags     : ['CTY_A1000M_Poids container', 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'CTY_A1000M_Titre VA AC', 'CTY_A1000M_Numéro Container', 'CTY_A1000R_Poids container', 'CTY_A1000R_Teneur arr. Vit. A (UV)', 'CTY_A1000R_Titre VA AC', 'CTY_A1000R_Numéro Container']
Date de début     : 2010-01-01
Date de fin       : 2025-12-16
Intervalle        : PT20M
Heure de début    : 00
Heure de fin      : 23
------------------------------------------------------------
DataFrame brut (df)   : Non chargé
DataFrame travail (data) : Non chargé
Données chargées : 419688 lignes, 8 colonnes
IN

KeyError: 'A1000M_Poids'